In [ ]:
# 경로 변경
%cd /content/drive/MyDrive/실전프로젝트

In [ ]:
# whisper 설치
!pip install -q faster-whisper

In [ ]:
# GPU 확인

import torch

print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Whisper large-turbo int8_float16 실행

from faster_whisper import WhisperModel

model = WhisperModel(
    "ThomasG/faster-whisper-large-v3-turbo-int8-fp16",
    device="cuda",
    compute_type="int8_float16"
)

segments, info = model.transcribe(
    "/content/drive/MyDrive/실전프로젝트/데이터 분석/data/New_Sample(노인남여)/원천데이터/1.AI챗봇/1.AI챗봇_1_자유대화(노인남여)_TRAINING/노인남여_노인대화07_F_1522434093_60_경상_실내/노인남여_노인대화07_F_1522434093_60_경상_실내_08580.wav",
    language="ko",
    beam_size=5
)

text = " ".join(segment.text.strip() for segment in segments)
print(text)

In [ ]:
# 데이터 파일 경로 모으기

import os

base_dir = "/content/drive/MyDrive/실전프로젝트/데이터 분석/data/New_Sample(노인남여)/"  # 데이터가 들어있는 상위 폴더 경로로 바꿔도 됨

wav_files = []
json_files = []

for root, dirs, files in os.walk(base_dir):
    for file in files:
        path = os.path.join(root, file)
        if file.lower().endswith(".wav"):
            wav_files.append(path)
        elif file.lower().endswith(".json"):
            json_files.append(path)

print("WAV 개수:", len(wav_files))
print("JSON 개수:", len(json_files))

print("\nWAV 예시")
for p in wav_files[:5]:
    print(p)

print("\nJSON 예시")
for p in json_files[:5]:
    print(p)

In [ ]:
import json
import pandas as pd
import os

rows = []

for json_path in json_files[:10]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    발화정보 = data.get("발화정보", {})
    대화정보 = data.get("대화정보", {})
    녹음자정보 = data.get("녹음자정보", {})

    rows.append({
        "json_path": json_path,
        "fileNm": 발화정보.get("fileNm"),
        "stt": 발화정보.get("stt"),
        "recrdTime": 발화정보.get("recrdTime"),
        "recrdQuality": 발화정보.get("recrdQuality"),
        "convrsThema": 대화정보.get("convrsThema"),
        "recrdEnvrn": 대화정보.get("recrdEnvrn"),
        "cityCode": 대화정보.get("cityCode"),
        "gender": 녹음자정보.get("gender"),
        "age": 녹음자정보.get("age"),
    })

df = pd.DataFrame(rows)
df

In [ ]:
wav_name_to_path = {os.path.basename(p): p for p in wav_files}

matched_rows = []

for json_path in json_files[:20]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    file_nm = data["발화정보"]["fileNm"]
    stt_text = data["발화정보"]["stt"].strip()

    matched_rows.append({
        "json_path": json_path,
        "fileNm": file_nm,
        "wav_exists": file_nm in wav_name_to_path,
        "wav_path": wav_name_to_path.get(file_nm),
        "stt": stt_text,
        "age": data["녹음자정보"].get("age"),
        "gender": data["녹음자정보"].get("gender"),
        "region": data["대화정보"].get("cityCode"),
        "env": data["대화정보"].get("recrdEnvrn"),
    })

match_df = pd.DataFrame(matched_rows)
match_df

In [ ]:
print("WAV 개수:", len(wav_files))
print("JSON 개수:", len(json_files))

print("WAV 예시:")
for p in wav_files[:10]:
    print(p)

print("JSON 예시:")
for p in json_files[:10]:
    print(p)

In [ ]:
import os, json

print("WAV 개수:", len(wav_files))
print("JSON 개수:", len(json_files))

print("\n[JSON fileNm 예시]")
for json_path in json_files[:5]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(repr(data["발화정보"]["fileNm"]))

print("\n[실제 WAV 파일명 예시]")
for wav_path in wav_files[:5]:
    print(repr(os.path.basename(wav_path)))

In [ ]:
import os
import json
import unicodedata
import pandas as pd
import time

def norm_name(name):
    base = os.path.basename(str(name)).strip()
    base = unicodedata.normalize("NFC", base)
    return base.lower()

start = time.time()

wav_name_to_path = {
    norm_name(p): p
    for p in wav_files
}

matched_rows = []

for idx, json_path in enumerate(json_files):
    if idx % 500 == 0:
        print(idx, "개 처리 중...")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    file_nm = data["발화정보"]["fileNm"]
    key = norm_name(file_nm)

    matched_rows.append({
        "json_path": json_path,
        "fileNm": file_nm,
        "normalized_fileNm": key,
        "wav_exists": key in wav_name_to_path,
        "wav_path": wav_name_to_path.get(key),
        "stt": data["발화정보"]["stt"].strip(),
        "age": data["녹음자정보"].get("age"),
        "gender": data["녹음자정보"].get("gender"),
        "region": data["대화정보"].get("cityCode"),
        "env": data["대화정보"].get("recrdEnvrn"),
        "topic": data["대화정보"].get("convrsThema"),
    })

match_df = pd.DataFrame(matched_rows)

print("완료")
print("걸린 시간:", round(time.time() - start, 2), "초")
print("전체 JSON:", len(match_df))
print("매칭 성공:", match_df["wav_exists"].sum())
print("매칭 실패:", (~match_df["wav_exists"]).sum())

match_df.head()

In [ ]:
import os
import json
import unicodedata
import pandas as pd
import time

def norm_name(name):
    base = os.path.basename(str(name)).strip()
    base = unicodedata.normalize("NFC", base)
    return base.lower()

start = time.time()

wav_name_to_path = {
    norm_name(p): p
    for p in wav_files
}

matched_rows = []

for json_path in json_files[:20]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    file_nm = data["발화정보"]["fileNm"]
    key = norm_name(file_nm)

    matched_rows.append({
        "json_path": json_path,
        "fileNm": file_nm,
        "normalized_fileNm": key,
        "wav_exists": key in wav_name_to_path,
        "wav_path": wav_name_to_path.get(key),
        "stt": data["발화정보"]["stt"].strip(),
        "age": data["녹음자정보"].get("age"),
        "gender": data["녹음자정보"].get("gender"),
        "region": data["대화정보"].get("cityCode"),
        "env": data["대화정보"].get("recrdEnvrn"),
        "topic": data["대화정보"].get("convrsThema"),
    })

match_df_test = pd.DataFrame(matched_rows)

print("걸린 시간:", round(time.time() - start, 2), "초")
print("매칭 성공:", match_df_test["wav_exists"].sum())
print("매칭 실패:", (~match_df_test["wav_exists"]).sum())

match_df_test

In [ ]:
LIMIT = 300

matched_rows = []

for idx, json_path in enumerate(json_files[:LIMIT]):
    if idx % 50 == 0:
        print(idx, "개 처리 중...")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    file_nm = data["발화정보"]["fileNm"]
    key = norm_name(file_nm)

    matched_rows.append({
        "json_path": json_path,
        "fileNm": file_nm,
        "wav_exists": key in wav_name_to_path,
        "wav_path": wav_name_to_path.get(key),
        "stt": data["발화정보"]["stt"].strip(),
        "age": data["녹음자정보"].get("age"),
        "gender": data["녹음자정보"].get("gender"),
        "region": data["대화정보"].get("cityCode"),
        "env": data["대화정보"].get("recrdEnvrn"),
        "topic": data["대화정보"].get("convrsThema"),
    })

match_df = pd.DataFrame(matched_rows)

print("처리 개수:", len(match_df))
print("매칭 성공:", match_df["wav_exists"].sum())
print("매칭 실패:", (~match_df["wav_exists"]).sum())

match_df.head()

In [ ]:
import time

for idx, json_path in enumerate(json_files[:1500]):
    start = time.time()

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print("오류 파일:", json_path)
        print(e)
        continue

    elapsed = time.time() - start

    if elapsed > 1:
        print("느린 파일:", idx, json_path, "걸린 시간:", round(elapsed, 2), "초")

    if idx % 100 == 0:
        print(idx, "개 처리")

In [ ]:
test_samples = match_df[match_df["wav_exists"] == True].head(3)

for i, row in test_samples.iterrows():
    print("WAV:", row["wav_path"])
    print("기준 전사문:", row["stt"])
    print("나이/성별/지역:", row["age"], row["gender"], row["region"])
    print("녹음환경/주제:", row["env"], row["topic"])
    print("-" * 50)

In [ ]:
!pip install -q faster-whisper

In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel(
    "ThomasG/faster-whisper-large-v3-turbo-int8-fp16",
    device="cuda",
    compute_type="int8_float16"
)

In [ ]:
import difflib
import pandas as pd

stt_results = []

for i, row in test_samples.iterrows():
    audio_path = row["wav_path"]
    reference = row["stt"]

    segments, info = model.transcribe(
        audio_path,
        language="ko",
        beam_size=5
    )

    pred = " ".join(seg.text.strip() for seg in segments).strip()
    similarity = difflib.SequenceMatcher(None, reference, pred).ratio()

    stt_results.append({
        "fileNm": row["fileNm"],
        "age": row["age"],
        "gender": row["gender"],
        "region": row["region"],
        "reference_stt": reference,
        "whisper_result": pred,
        "similarity": round(similarity, 3)
    })

stt_result_df = pd.DataFrame(stt_results)
stt_result_df